# TP 01 - Blockchain
**Hachage SHA-256, chaînage de blocs et minage**

Université Mundiapolis - Filière Ingénierie Informatique

## 1. Hachage SHA-256

### 1.1 Fonction `calculer_sha256`

In [1]:
import hashlib

def calculer_sha256(texte):
    empreinte = hashlib.sha256(texte.encode("utf-8")).hexdigest()
    return empreinte

print(calculer_sha256("Blockchain"))

625da44e4eaf58d61cf048d168aa6f5e492dea166d8bb54ec06c30de07db57e1


### 1.2 Déterminisme : même texte, même empreinte

In [2]:
h1 = calculer_sha256("Blockchain")
h2 = calculer_sha256("Blockchain")
print(h1)
print(h2)
print("Identiques ?", h1 == h2)

625da44e4eaf58d61cf048d168aa6f5e492dea166d8bb54ec06c30de07db57e1
625da44e4eaf58d61cf048d168aa6f5e492dea166d8bb54ec06c30de07db57e1
Identiques ? True


### 1.3 Longueur de l'empreinte

In [3]:
for texte in ["a", "Mundiapolis", "x" * 1000]:
    print(len(texte), "caracteres ->", len(calculer_sha256(texte)))

1 caracteres -> 64
11 caracteres -> 64
1000 caracteres -> 64


### 1.4 Effet avalanche : fonction `compter_differences`

In [4]:
def compter_differences(h1, h2):
    nb_differences = 0
    for i in range(len(h1)):
        if h1[i] != h2[i]:
            nb_differences += 1
    return nb_differences

h1 = calculer_sha256("Bloc numero 1")
h2 = calculer_sha256("Bloc numero 2")
print(h1)
print(h2)
print(compter_differences(h1, h2), "caracteres differents sur", len(h1))

c17a909c42cb7d2acfb192bb9f75207a83cac136d2a006104a62b482b8b58793
587fa627c8742464522772db52e48634c1f40b5a4b7fc7e4a006c5f831059133
59 caracteres differents sur 64


### 1.5 Effet avalanche sur deux transactions presque identiques

In [5]:
h1 = calculer_sha256("Ali envoie 12$ a Sara")
h2 = calculer_sha256("Ali envoie 13$ a Sara")
print(h1)
print(h2)
print(compter_differences(h1, h2), "caracteres differents sur", len(h1))

572704069428e74fa353c4dd4a370f99f0af2ce99e6d755d2745e2eeab0aa538
02c23a118142da8378290b814975c081671720986bda11b7689d1c8d7179ecc1
62 caracteres differents sur 64


## 2. Représenter un bloc et chaîner

Registre des transactions utilisé dans tout le TP :

In [6]:
donnees_blocs = [
    "Bloc de genese Mundiapolis",
    "Ali envoie 12$ a Sara",
    "Sara envoie 5$ a Karim",
    "Karim envoie 3$ a Nadia",
    "Nadia envoie 7$ a Ali",
]

### 2.1 Fonction `creer_bloc`

In [7]:
def creer_bloc(index, donnees, hash_precedent):
    bloc = {
        "index": index,
        "donnees": donnees,
        "nonce": 0,
        "hash_precedent": hash_precedent,
        "hash": ""
    }
    return bloc

print(creer_bloc(1, "Ali envoie 12$ a Sara", "abc"))

{'index': 1, 'donnees': 'Ali envoie 12$ a Sara', 'nonce': 0, 'hash_precedent': 'abc', 'hash': ''}


### 2.2 Fonction `calculer_hash_bloc`

In [8]:
def calculer_hash_bloc(bloc):
    contenu = str(bloc["index"]) + str(bloc["nonce"]) + bloc["donnees"] + bloc["hash_precedent"]
    return calculer_sha256(contenu)

### 2.3 Bloc de genèse

In [9]:
genese = creer_bloc(0, "Bloc de genese Mundiapolis", "0" * 64)
genese["hash"] = calculer_hash_bloc(genese)
chaine = [genese] # ces 3 lignes de code successives pour imaginer qu'on cree une blockchain qui est se forme d'un seul bloc pour le moment, c'est le bloc genese
# print(chaine)
# print(chaine[0])
print("Hash de la genese :", genese["hash"])

Hash de la genese : fb75966eb8d3dda427fec6a298feed4a9d5b0973343ba8a697e710a04f101599


### 2.4 Fonction `ajouter_bloc`

In [10]:
def ajouter_bloc(chaine, donnees):
    dernier_bloc = chaine[len(chaine) - 1]
    nouvel_bloc = creer_bloc(len(chaine), donnees, dernier_bloc["hash"])
    nouvel_bloc["hash"] = calculer_hash_bloc(nouvel_bloc)
    chaine.append(nouvel_bloc)

### 2.5 Ajout des quatre transactions

In [11]:
for i in range(1, len(donnees_blocs)):
    ajouter_bloc(chaine, donnees_blocs[i])

for bloc in chaine:
    print(f"Bloc {bloc['index']} | prec : {bloc['hash_precedent'][0:10]} | hash : {bloc['hash'][0:10]}")

Bloc 0 | prec : 0000000000 | hash : fb75966eb8
Bloc 1 | prec : fb75966eb8 | hash : 7f6af0a93a
Bloc 2 | prec : 7f6af0a93a | hash : f9adf24efc
Bloc 3 | prec : f9adf24efc | hash : 643f7518cb
Bloc 4 | prec : 643f7518cb | hash : 34b442a5b6


### 2.6 Fonction `afficher_bloc`

In [12]:
def afficher_bloc(bloc):
    print("index          :", bloc["index"])
    print("donnees        :", bloc["donnees"])
    print("nonce          :", bloc["nonce"])
    print("hash_precedent :", bloc["hash_precedent"])
    print("hash           :", bloc["hash"])

afficher_bloc(chaine[1])

index          : 1
donnees        : Ali envoie 12$ a Sara
nonce          : 0
hash_precedent : fb75966eb8d3dda427fec6a298feed4a9d5b0973343ba8a697e710a04f101599
hash           : 7f6af0a93ab7d810af731ff70ddc8c093d133a826c2bce7a2e73212c393ebff4


In [13]:
def lire_transaction(donnees):
    mots = donnees.split(" ")
    if len(mots) != 5 or mots[1] != "envoie" or mots[2][-1] != "$" or mots[3] != "a":
        return None
    emetteur = mots[0]
    montant = int(mots[2].replace("$", ""))
    beneficiaire = mots[4]
    return (emetteur, montant, beneficiaire)
print(lire_transaction("Ali envoie 12$ a Sara"))

('Ali', 12, 'Sara')


### 2.7 Fonction `lire_transaction`

In [14]:
def lire_transaction(donnees):
    mots = donnees.split(" ")
    emetteur = mots[0]
    montant = int(mots[2].replace("$", ""))
    beneficiaire = mots[4]
    return emetteur, montant, beneficiaire

print(lire_transaction("Ali envoie 12$ a Sara"))

('Ali', 12, 'Sara')


### 2.8 Fonction `calculer_soldes`

In [15]:
def calculer_soldes(chaine):
    soldes = {"Ali": 20, "Sara": 20, "Karim": 20, "Nadia": 20}
    for i in range(1, len(chaine)):
        emetteur, montant, beneficiaire = lire_transaction(chaine[i]["donnees"])
        soldes[emetteur] -= montant
        soldes[beneficiaire] += montant
    return soldes

print(calculer_soldes(chaine))

{'Ali': 15, 'Sara': 27, 'Karim': 22, 'Nadia': 16}


## 3. Minage : preuve de travail

### 3.1 Fonction `miner_bloc`

In [16]:
def miner_bloc(bloc, difficulte):
    cible = "0" * difficulte
    bloc["nonce"] = 0
    bloc["hash"] = calculer_hash_bloc(bloc)
    essais = 1
    while not bloc["hash"].startswith(cible):
        bloc["nonce"] += 1
        bloc["hash"] = calculer_hash_bloc(bloc)
        essais += 1
    return essais


### 3.2 Miner un nouveau bloc de genèse (difficulté 4)

In [17]:
# On crée un nouveau bloc de genèse (comme demandé dans l'énoncé) au lieu de réutiliser
# celui de la question 2.3. miner_bloc modifie le bloc directement (nonce et hash), et
# le genese de 2.3 est aussi chaine[0] : le miner changerait son hash sans mettre à jour
# le hash_precedent du bloc 1, ce qui casserait la chaîne de la partie 2.
genese = creer_bloc(0, "Bloc de genese Mundiapolis", "0" * 64)
#genese["hash"] = calculer_hash_bloc(genese) #cette ligne n'est pas necessaire ici, car 
# dans miner_bloc : si le test startswith passe avant tout calcul, un hash périmé ou vide 
# pose problème. Ici, on calcule toujours avant de tester.
essais = miner_bloc(genese, 4)
print("Nonce trouve :", genese["nonce"])
print("Essais       :", essais)
print("Hash         :", genese["hash"])

Nonce trouve : 41489
Essais       : 41490
Hash         : 0000eb2c58b542efffaf1dcdada2baf8832a9bb933949c41442410ec8a9e7889


### 3.3 Fonction `construire_chaine`

In [18]:
def construire_chaine(liste_donnees, difficulte):
    chaine = []
    hash_precedent = "0" * 64
    for i in range(len(liste_donnees)):
        bloc = creer_bloc(i, liste_donnees[i], hash_precedent)
        essais = miner_bloc(bloc, difficulte)
        print(f"[mine] Bloc {i} : {essais} essais -> {bloc['hash'][0:20]}...")
        chaine.append(bloc)
        hash_precedent = bloc["hash"]
    return chaine
#3.3 reprend l’idée de 2.4, qui est de construire une chaîne en chaînant 
# chaque bloc au précédent. La différence n’est pas seulement 
# « une donnée à la fois » contre « une liste » : 3.3 mine chaque bloc, 
# et c’est ce qui change la nature de la chaîne.

### 3.4 Miner les cinq blocs du registre

In [19]:
chaine = construire_chaine(donnees_blocs, 4)

[mine] Bloc 0 : 41490 essais -> 0000eb2c58b542efffaf...
[mine] Bloc 1 : 48145 essais -> 0000fb41e6cae14ae3b7...
[mine] Bloc 2 : 22968 essais -> 00008ddc86166eaece69...
[mine] Bloc 3 : 13919 essais -> 0000cc4bf72544ba469c...
[mine] Bloc 4 : 20150 essais -> 00009538d1e384c589b8...


## 4. Visualiser et vérifier la chaîne

### 4.1 Fonction `afficher_chaine`

In [20]:
def afficher_chaine(chaine):
    for i in range(len(chaine)):
        bloc = chaine[i]
        print("+" + "-" * 48 + "+")
        print("| BLOC", bloc["index"])
        print("+" + "-" * 48 + "+")
        print("| Donnees      :", bloc["donnees"])
        print("| Nonce        :", bloc["nonce"])
        print("| Hash prec.   :", bloc["hash_precedent"][0:28] + "...")
        print("| Hash du bloc :", bloc["hash"][0:28] + "...")
        print("+" + "-" * 48 + "+")
        if i < len(chaine) - 1:
            print("                        |")
            print("                        v")

afficher_chaine(chaine)

+------------------------------------------------+
| BLOC 0
+------------------------------------------------+
| Donnees      : Bloc de genese Mundiapolis
| Nonce        : 41489
| Hash prec.   : 0000000000000000000000000000...
| Hash du bloc : 0000eb2c58b542efffaf1dcdada2...
+------------------------------------------------+
                        |
                        v
+------------------------------------------------+
| BLOC 1
+------------------------------------------------+
| Donnees      : Ali envoie 12$ a Sara
| Nonce        : 48144
| Hash prec.   : 0000eb2c58b542efffaf1dcdada2...
| Hash du bloc : 0000fb41e6cae14ae3b79dfec78a...
+------------------------------------------------+
                        |
                        v
+------------------------------------------------+
| BLOC 2
+------------------------------------------------+
| Donnees      : Sara envoie 5$ a Karim
| Nonce        : 22967
| Hash prec.   : 0000fb41e6cae14ae3b79dfec78a...
| Hash du bloc : 00008dd

### 4.2 Fonction `verifier_chaine`

In [21]:
def verifier_chaine(chaine, difficulte):
    cible = "0" * difficulte
    for i in range(len(chaine)):
        bloc = chaine[i]
        if calculer_hash_bloc(bloc) != bloc["hash"]:
            print(f"Bloc {i} : le hash recalculee ne correspond pas au hash stockee")
            return False
        if not bloc["hash"].startswith(cible):
            print(f"Bloc {i} : la preuve de travail n'est pas satisfaite")
            return False
        if i > 0 and bloc["hash_precedent"] != chaine[i - 1]["hash"]:
            print(f"Bloc {i} : hash_precedent different du hash du bloc {i - 1}")
            return False
    return True

### 4.3 Vérification de la chaîne

In [22]:
print("Chaine valide ?", verifier_chaine(chaine, 4))

Chaine valide ? True


## 5. Augmenter la difficulté

### 5.1 Fonction `compter_essais`

In [23]:
def compter_essais(chaine):
    total = 0
    for bloc in chaine:
        total += bloc["nonce"] + 1
    return total

### 5.2 Mesure pour la difficulté 4

In [24]:
import time

debut = time.time()
chaine4 = construire_chaine(donnees_blocs, 4)
duree4 = time.time() - debut
total4 = compter_essais(chaine4)
print("Total :", total4, "essais en", round(duree4, 2), "s")

[mine] Bloc 0 : 41490 essais -> 0000eb2c58b542efffaf...
[mine] Bloc 1 : 48145 essais -> 0000fb41e6cae14ae3b7...
[mine] Bloc 2 : 22968 essais -> 00008ddc86166eaece69...
[mine] Bloc 3 : 13919 essais -> 0000cc4bf72544ba469c...
[mine] Bloc 4 : 20150 essais -> 00009538d1e384c589b8...
Total : 146672 essais en 0.27 s


### 5.3 Mesure pour la difficulté 5

In [25]:
debut = time.time()
chaine5 = construire_chaine(donnees_blocs, 5)
duree5 = time.time() - debut
total5 = compter_essais(chaine5)
print("Total :", total5, "essais en", round(duree5, 2), "s")

[mine] Bloc 0 : 985225 essais -> 00000527236c51741ed5...
[mine] Bloc 1 : 272033 essais -> 00000af5c35071593b42...
[mine] Bloc 2 : 169506 essais -> 000004ba3e46a356355c...
[mine] Bloc 3 : 446240 essais -> 00000edcbadc3c1dda49...
[mine] Bloc 4 : 355141 essais -> 00000ae0d4a0deac94f5...
Total : 2228145 essais en 3.04 s


### 5.4 Rapport, débit et estimation pour 8 zéros

In [26]:
rapport = total5 / total4
print("Rapport des essais : ", round(rapport, 1))

debit = total5 / duree5
print("Debit : ", round(debit, 1), "hash/s")

estimation_secondes = 16**8 / debit
estimation_heures = estimation_secondes / 3600
print("Estimation du temps pour 8 zeros : ", round(estimation_secondes, 1), 
      "s, soit ", round(estimation_heures, 1), " h")

Rapport des essais :  15.2
Debit :  733236.0 hash/s
Estimation du temps pour 8 zeros :  5857.6 s, soit  1.6  h


### 5.5 Tableau récapitulatif
Regardez la reponse dans le tableau du document pdf :)

## 6. Falsifier un montant

### 6.1 Falsification sans rien recalculer

In [27]:
chaine = construire_chaine(donnees_blocs, 4)
print("Chaine valide ?", verifier_chaine(chaine, 4))

chaine[1]["donnees"] = "Ali envoie 2500$ a Sara"
print("Chaine valide ?", verifier_chaine(chaine, 4))

[mine] Bloc 0 : 41490 essais -> 0000eb2c58b542efffaf...
[mine] Bloc 1 : 48145 essais -> 0000fb41e6cae14ae3b7...
[mine] Bloc 2 : 22968 essais -> 00008ddc86166eaece69...
[mine] Bloc 3 : 13919 essais -> 0000cc4bf72544ba469c...
[mine] Bloc 4 : 20150 essais -> 00009538d1e384c589b8...
Chaine valide ? True
Bloc 1 : le hash recalculee ne correspond pas au hash stockee
Chaine valide ? False


### 6.2 Hash stocké et hash recalculé du bloc 1

In [28]:
print("hash stocke    :", chaine[1]["hash"])
print("hash recalcule :", calculer_hash_bloc(chaine[1]))

hash stocke    : 0000fb41e6cae14ae3b79dfec78a52e6504a09d7e3cbe8aba28b9263eaa215a9
hash recalcule : 8e265235c949f39c7809bf18bc9df3d44e8fe463def4471654281f38e1e57e6d


### 6.3 Re-miner seulement le bloc 1

In [29]:
essais = miner_bloc(chaine[1], 4)
print(f"Bloc 1 re-mine en {essais} essais -> {chaine[1]['hash'][0:20]}...")
print("Chaine valide ?", verifier_chaine(chaine, 4))

Bloc 1 re-mine en 31895 essais -> 0000f5c9f0d67d186e41...
Bloc 2 : hash_precedent different du hash du bloc 1
Chaine valide ? False


### 6.4 Fonction `reminer_a_partir_de`

In [30]:
def reminer_a_partir_de(chaine, debut, difficulte):
    total = 0
    for i in range(debut, len(chaine)):
        if i > 0:
            chaine[i]["hash_precedent"] = chaine[i - 1]["hash"]
        essais = miner_bloc(chaine[i], difficulte)
        print(f"[re-mine] Bloc {i} : {essais} essais -> {chaine[i]['hash'][0:20]}...")
        total += essais
    return total

### 6.5 Falsifier le bloc 1 puis re-miner toute la fin de la chaîne

In [31]:
chaine = construire_chaine(donnees_blocs, 4)
chaine[1]["donnees"] = "Ali envoie 2500$ a Sara"
cout_bloc1 = reminer_a_partir_de(chaine, 1, 4)
print("Total :", cout_bloc1, "essais")
print("Chaine valide ?", verifier_chaine(chaine, 4))
afficher_chaine(chaine)

[mine] Bloc 0 : 41490 essais -> 0000eb2c58b542efffaf...
[mine] Bloc 1 : 48145 essais -> 0000fb41e6cae14ae3b7...
[mine] Bloc 2 : 22968 essais -> 00008ddc86166eaece69...
[mine] Bloc 3 : 13919 essais -> 0000cc4bf72544ba469c...
[mine] Bloc 4 : 20150 essais -> 00009538d1e384c589b8...
[re-mine] Bloc 1 : 31895 essais -> 0000f5c9f0d67d186e41...
[re-mine] Bloc 2 : 32567 essais -> 00003952799ef40f2394...
[re-mine] Bloc 3 : 5397 essais -> 00004ff8de4bc444ef57...
[re-mine] Bloc 4 : 78354 essais -> 000042c383bce62f9ca8...
Total : 148213 essais
Chaine valide ? True
+------------------------------------------------+
| BLOC 0
+------------------------------------------------+
| Donnees      : Bloc de genese Mundiapolis
| Nonce        : 41489
| Hash prec.   : 0000000000000000000000000000...
| Hash du bloc : 0000eb2c58b542efffaf1dcdada2...
+------------------------------------------------+
                        |
                        v
+------------------------------------------------+
| BLOC 1
+--

### 6.6 Falsifier le dernier bloc

In [32]:
chaine = construire_chaine(donnees_blocs, 4)
chaine[4]["donnees"] = "Nadia envoie 700$ a Ali"
cout_bloc4 = reminer_a_partir_de(chaine, 4, 4)
print("Falsifier le bloc 1 :", cout_bloc1, "essais")
print("Falsifier le bloc 4 :", cout_bloc4, "essais")

[mine] Bloc 0 : 41490 essais -> 0000eb2c58b542efffaf...
[mine] Bloc 1 : 48145 essais -> 0000fb41e6cae14ae3b7...
[mine] Bloc 2 : 22968 essais -> 00008ddc86166eaece69...
[mine] Bloc 3 : 13919 essais -> 0000cc4bf72544ba469c...
[mine] Bloc 4 : 20150 essais -> 00009538d1e384c589b8...
[re-mine] Bloc 4 : 29829 essais -> 0000086e30076da93b8d...
Falsifier le bloc 1 : 148213 essais
Falsifier le bloc 4 : 29829 essais


### 6.7 Même falsification avec la difficulté 5

In [33]:
chaine = construire_chaine(donnees_blocs, 5)
chaine[1]["donnees"] = "Ali envoie 2500$ a Sara"
cout_bloc1_d5 = reminer_a_partir_de(chaine, 1, 5)
print("Difficulte 4 :", cout_bloc1, "essais")
print("Difficulte 5 :", cout_bloc1_d5, "essais")
print("Facteur      :", round(cout_bloc1_d5 / cout_bloc1, 1))

[mine] Bloc 0 : 985225 essais -> 00000527236c51741ed5...
[mine] Bloc 1 : 272033 essais -> 00000af5c35071593b42...
[mine] Bloc 2 : 169506 essais -> 000004ba3e46a356355c...
[mine] Bloc 3 : 446240 essais -> 00000edcbadc3c1dda49...
[mine] Bloc 4 : 355141 essais -> 00000ae0d4a0deac94f5...
[re-mine] Bloc 1 : 855060 essais -> 00000413c0e7f1d47eeb...
[re-mine] Bloc 2 : 1202634 essais -> 00000336e28689cda4ba...
[re-mine] Bloc 3 : 176401 essais -> 000001e7cdf2a731fc76...
[re-mine] Bloc 4 : 198623 essais -> 00000a619dace353cf4f...
Difficulte 4 : 148213 essais
Difficulte 5 : 2432718 essais
Facteur      : 16.4


## Conclusion

Changer un seul caractère du montant fait perdre au hash ses zéros de tête. Après re-minage, le bloc 1 possède un nouveau hash que le bloc 2 ne référence pas : il faut re-miner le bloc 2, puis le bloc 3, et ainsi de suite. Falsifier une transaction ancienne coûte donc le re-minage de toute la fin de la chaîne, alors que falsifier la dernière ne coûte qu'un seul bloc. Avec un zéro de plus dans la cible, ce coût est multiplié par 16 environ.